In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import heapq
import random

# Reproducibility: Fixing seeds ensures the dataset can be regenerated consistently for evaluation and experimentation.

np.random.seed(42)
random.seed(42)

# GLOBAL SIMULATION CONFIGURATION (CONTROLLED AGGRESSIVE)
# These parameters control scale, congestion intensity,and temporal resolution of the synthetic environment.

N_RESTAURANTS = 200       # Total restuarants in ecosystem
DAYS = 30
TIME_SLOT_MIN = 5
START_DATE = datetime(2024, 1, 1)

BASE_ARRIVAL_RATE = 0.25  # Base demand intensity (tuned to create realistic congestion)


# CUISINE CONFIGURATION (REALISTIC RANGES)
# Each cuisine has a realistic base prep-time range.
# This introduces heterogeneity across restaurants.

cuisine_config = {
    "Fast Food": (8, 14),
    "Cafe": (10, 16),
    "North Indian": (15, 25),
    "South Indian": (12, 20),
    "Chinese": (14, 22),
    "Biryani": (18, 32),
    "Pizza": (14, 22),
    "Continental": (20, 35),
    "Desserts": (8, 16)
}

# Weather configuration
# Weather affects both demand and prep complexity.
# Rain & storms increase demand and slow operations.

weather_types = ["Sunny", "Cloudy", "Rainy", "Stormy"]

weather_demand_factor = {
    "Sunny": 1.0,
    "Cloudy": 1.05,
    "Rainy": 1.15,
    "Stormy": 1.25
}

weather_probs = [0.6, 0.25, 0.12, 0.03]

# RESTAURANT CREATION
# Simulates real-world variation in capacity, efficiency, ratings and popularity.
# Designed to create urban stress conditions.

restaurants = []

for i in range(N_RESTAURANTS):

    cuisine = random.choice(list(cuisine_config.keys()))
    base_low, base_high = cuisine_config[cuisine]

    rating = np.round(np.random.uniform(3.0, 5.0), 1)

    # Fewer chefs → more congestion variability
    chef_count = np.random.randint(3, 10)

    productivity = np.random.uniform(1.6, 2.2)
    capacity = int(chef_count * productivity)

    base_prep = np.random.randint(base_low, base_high)

    # Efficiency improves with rating but has randomness
    efficiency = np.clip(
        0.75 + (rating - 3.0) * 0.2 + np.random.normal(0, 0.04),
        0.75, 1.25
    )

    # Higher-rated restaurants get more demand
    popularity_score = (rating ** 3) * np.random.uniform(0.7, 1.3)

    restaurants.append({
        "restaurant_id": i,
        "cuisine_type": cuisine,
        "base_prep_time": base_prep,
        "restaurant_capacity": capacity,
        "restaurant_rating": rating,
        "efficiency_score": efficiency,
        "chef_count": chef_count,
        "popularity_score": popularity_score
    })

restaurants_df = pd.DataFrame(restaurants)

# Normalize popularity to create probabilistic restaurant selection
popularity_probs = restaurants_df["popularity_score"]
popularity_probs = popularity_probs / popularity_probs.sum()

# TIME-BASED DEMAND MULTIPLIER
# Captures lunch rush, dinner rush, low-demand hours.

def time_demand_multiplier(hour):
    if 12 <= hour <= 14:
        return 1.8
    elif 19 <= hour <= 22:
        return 2.2
    elif 15 <= hour <= 17:
        return 0.7
    elif hour >= 23 or hour <= 5:
        return 0.5
    else:
        return 1.0

# SIMULATION STATE
# Each restaurant maintains a min-heap tracking completion times of active orders.
# This models real kitchen queue processing.

active_heaps = {i: [] for i in range(N_RESTAURANTS)}
data = []

# MAIN SIMULATION LOOP
# Simulates day-by-day, 5-minute interval operations.

for day in range(DAYS):

    for minute_block in range(0, 24 * 60, TIME_SLOT_MIN):

        current_time = START_DATE + timedelta(days=day, minutes=minute_block)
        hour = current_time.hour
        weekend = current_time.weekday() >= 5

        # Sample weather and festival conditions
        weather = np.random.choice(weather_types, p=weather_probs)
        festival_flag = np.random.choice([0,1], p=[0.92,0.08])

        demand_multiplier = time_demand_multiplier(hour)

        if weekend:
            demand_multiplier *= 1.25

        demand_multiplier *= weather_demand_factor[weather]

        if festival_flag:
            demand_multiplier *= 1.4

        # Orders follow Poisson arrival (standard queueing assumption)
        total_orders_this_slot = np.random.poisson(
            BASE_ARRIVAL_RATE * demand_multiplier * N_RESTAURANTS
        )

        if total_orders_this_slot == 0:
            continue

        # Restaurants selected proportional to popularity
        chosen_restaurants = np.random.choice(
            restaurants_df["restaurant_id"],
            size=total_orders_this_slot,
            p=popularity_probs
        )

        for rest_id in chosen_restaurants:

            rest = restaurants_df.iloc[rest_id]
            heap = active_heaps[rest_id]

            # Remove completed orders from heap
            while heap and heap[0] <= current_time:
                heapq.heappop(heap)

            active_orders = len(heap)

            # Order-level variability
            total_items = np.random.randint(1, 7)
            item_complexity = total_items * np.random.uniform(1.0, 2.5)

            kitchen_util = active_orders / rest.restaurant_capacity

            # Queue starts forming when kitchen is overloaded
            queue_length = max(
                active_orders - 0.6 * rest.restaurant_capacity,
                0
            )

            # PREP TIME CALCULATION (AGGRESSIVE BUT CONTROLLED)
            # Models nonlinear congestion, kitchen stress, order complexity, and random heavy-tail delays.

            prep = rest.base_prep_time
            prep += item_complexity * 1.3

            # Stronger nonlinear congestion
            prep += (queue_length ** 1.15) * 0.7
            prep += kitchen_util * 8
            prep -= rest.efficiency_score * 4

            if weather in ["Rainy","Stormy"]:
                prep *= 1.05

            # Rare extreme delay (heavy tail behavior)
            if np.random.rand() < 0.02:
                prep *= np.random.uniform(1.5, 2.2)

            prep += np.random.normal(0, 2)
            prep = max(5, prep)

            completion_time = current_time + timedelta(minutes=prep)
            heapq.heappush(heap, completion_time)

            # Store all operational features
            data.append({
                "order_time": current_time,
                "restaurant_id": rest_id,
                "cuisine_type": rest.cuisine_type,
                "restaurant_capacity": rest.restaurant_capacity,
                "restaurant_rating": rest.restaurant_rating,
                "chef_count": rest.chef_count,
                "efficiency_score": rest.efficiency_score,
                "total_items": total_items,
                "item_complexity_score": round(item_complexity,2),
                "active_orders_in_kitchen": active_orders,
                "queue_length": round(queue_length,2),
                "kitchen_utilization": round(kitchen_util,2),
                "weather": weather,
                "festival_flag": festival_flag,
                "hour": hour,
                "weekend": int(weekend),
                "prep_time": round(prep,2)
            })

# FINAL DATAFRAME
# Prints summary statistics for validation.

df = pd.DataFrame(data)

print("Total Orders:", len(df))
print("Average Prep Time:", round(df["prep_time"].mean(),2))
print("Median Prep Time:", round(df["prep_time"].median(),2))
print("90th Percentile:", round(np.percentile(df["prep_time"],90),2))
print("99th Percentile:", round(np.percentile(df["prep_time"],99),2))
print("Mean Queue Length:", round(df["queue_length"].mean(),2))

df.to_csv("zomato_kpt_dataset.csv", index=False)

print("Dataset generated successfully.")

Total Orders: 550502
Average Prep Time: 23.63
Median Prep Time: 22.06
90th Percentile: 35.79
99th Percentile: 57.57
Mean Queue Length: 0.38
Dataset generated successfully.


In [2]:
df.head()

,order_time,restaurant_id,cuisine_type,restaurant_capacity,restaurant_rating,chef_count,efficiency_score,total_items,item_complexity_score,active_orders_in_kitchen,queue_length,kitchen_utilization,weather,festival_flag,hour,weekend,prep_time
0,2024-01-01,37,Biryani,10,3.6,5,0.876097,4,6.24,0,0.0,0.0,Rainy,1,0,0,25.77
1,2024-01-01,198,Continental,13,3.0,8,0.812020,6,14.66,0,0.0,0.0,Rainy,1,0,0,55.89
2,2024-01-01,50,Fast Food,5,4.5,3,0.994893,4,4.46,0,0.0,0.0,Rainy,1,0,0,8.59
3,2024-01-01,187,Biryani,7,4.1,4,0.978308,5,11.80,0,0.0,0.0,Rainy,1,0,0,33.36
4,2024-01-01,129,North Indian,5,4.1,3,0.980710,2,3.63,0,0.0,0.0,Rainy,1,0,0,24.25


In [3]:
df.describe()

,order_time,restaurant_id,restaurant_capacity,restaurant_rating,chef_count,efficiency_score,total_items,item_complexity_score,active_orders_in_kitchen,queue_length,kitchen_utilization,festival_flag,hour,weekend,prep_time
count,550502,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000,550502.000000
mean,2024-01-16 04:26:08.301659136,100.682426,10.882722,4.270943,5.995388,1.007651,3.500214,6.129504,2.503064,0.379307,0.291748,0.109144,13.900894,0.312061,23.631440
min,2024-01-01 00:00:00,0.000000,4.000000,3.000000,3.000000,0.750000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000
25%,2024-01-08 13:55:00,52.000000,7.000000,3.800000,4.000000,0.920200,2.000000,3.310000,1.000000,0.000000,0.060000,0.000000,9.000000,0.000000,16.850000
50%,2024-01-15 23:35:00,97.000000,11.000000,4.400000,6.000000,1.032487,4.000000,5.740000,2.000000,0.000000,0.170000,0.000000,14.000000,0.000000,22.060000
75%,2024-01-23 14:40:00,149.000000,14.000000,4.700000,8.000000,1.103445,5.000000,8.520000,4.000000,0.000000,0.380000,0.000000,20.000000,1.000000,28.410000
max,2024-01-30 23:55:00,199.000000,19.000000,5.000000,9.000000,1.235295,6.000000,15.000000,37.000000,34.000000,8.500000,1.000000,23.000000,1.000000,228.030000
std,NaN,56.128945,4.249872,0.558938,2.113678,0.119339,1.706080,3.435258,2.688169,1.554340,0.431128,0.311820,6.296465,0.463335,10.210106


In [4]:
print(df.columns)

Index(['order_time', 'restaurant_id', 'cuisine_type', 'restaurant_capacity',
       'restaurant_rating', 'chef_count', 'efficiency_score', 'total_items',
       'item_complexity_score', 'active_orders_in_kitchen', 'queue_length',
       'kitchen_utilization', 'weather', 'festival_flag', 'hour', 'weekend',
       'prep_time'],
      dtype='object')
